# PointTransformerV2 — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.

⚠️ Requires `torch-scatter` + `torch-cluster` — install matching wheels from https://data.pyg.org/whl/

In [1]:
# ── Install (run once if needed) ─────────────────────────────────────────────
# pip install torch laspy open3d numpy scikit-learn joblib tqdm mlflow matplotlib
# For PTv1/PTv2/GNN also:
# pip install torch-scatter torch-cluster -f https://data.pyg.org/whl/torch-<VER>+<CUDA>.html

import os, glob, glob, copy, random, logging
import numpy as np
import torch
import torch.nn as nn
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Running on: {DEVICE}")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-07-11 03:02:40,085 | INFO | Running on: cuda


In [2]:
# ── Configuration — change values here, nowhere else ────────────────────────
CONFIG = {
    "train_dir"      : "../data/train",      # labelled .las files
    "test_dir"       : "../data/test",       # unlabelled .las files for inference
    "checkpoint_dir" : "../checkpoints_ptv2_voxel",  # NEW dir: voxel pipeline is a
                                                     # different data distribution —
                                                     # do NOT resume old checkpoints
    "num_classes"    : 2,                 # 0 = environment, 1 = wood powder
    "target_class"   : 1,                 # class we want to highlight (green)
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "seed"           : 42,
    # ── training ──
    "epochs"         : 150,
    "patience"       : 40,                # early stop if val mIoU doesn't improve
    "batch_size"     : 2,
    "num_points"     : 32768,             # points per training chunk
    "chunks_per_cloud": 8,
    "lr"             : 2e-4,
    "grad_accum"     : 8,    # gradient accumulation steps (effective batch = batch_size × grad_accum)
    "sched_factor"   : 0.5,  # LR scheduler: multiply LR by this on plateau
    "sched_patience" : 10,   # LR scheduler: epochs without val-mIoU improvement before reducing
    "min_lr"         : 1e-6, # LR scheduler: lower bound on LR
    "in_channels"    : 7,                # xyz + height + normals
    # ── preprocessing ──
    "voxel_size"     : 0.01,             # metres; 1 cm voxel downsample before
                                         # features (0 = disabled). 400k–800k
                                         # points → roughly 60k–150k, so a 32768-pt
                                         # chunk covers 20–50% of the cloud
                                         # instead of 2–4% → much more context.
    # ── visualization ──
    "max_vis_files"  : 0,                # how many test files to show in 3D
    # ── MLflow ──
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "PTV2_Segmentation",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
NUM_CLASSES = CONFIG["num_classes"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


In [3]:
# ── Point cloud loader ───────────────────────────────────────────────────────
def load_pointcloud(path):
    """Read a .las/.laz file. Returns (points[N,3], labels[N] or None)."""
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    return pts, labels

def list_files(folder):
    """Return sorted list of .las/.laz files in a folder."""
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

# ── Split labelled files into train / val / test ─────────────────────────────
all_files = list_files(CONFIG["train_dir"])
assert all_files, f"No .las files found in {CONFIG['train_dir']}"

rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_val  = max(1, int(len(all_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_files[i] for i in order[:n_val]]
TEST_FILES  = [all_files[i] for i in order[n_val : n_val + n_test]]
TRAIN_FILES = [all_files[i] for i in order[n_val + n_test :]]
INFER_FILES = list_files(CONFIG["test_dir"])   # no labels, final inference only

log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")


2026-07-11 03:02:40,113 | INFO | train=33  val=6  test=6  inference=18


In [4]:
# ── 7-channel feature computation ───────────────────────────────────────────
# Each point gets: normalized xyz (3) + height above floor (1) + surface normal (3)
def make_features(points):
    """Turn raw XYZ into 7-channel features. Returns float32 array (N,7)."""
    center = points.mean(axis=0, keepdims=True)
    scale  = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)

    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)

    # surface normals via Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(16))
    pcd.orient_normals_to_align_with_direction([0., 0., 1.])
    normals = np.asarray(pcd.normals, dtype=np.float32)

    return np.column_stack([norm_xyz, height[:, None], normals])

# ── Voxel downsampling (metric space, keeps labels) ──────────────────────────
def voxel_downsample_idx(points, voxel):
    """Return indices of ONE representative point per voxel — the point closest
    to its voxel's centroid. Works on raw metric xyz, so `voxel` is in metres.
    Keeping original indices (instead of averaged points) lets labels, normals
    and any other per-point data travel with the surviving points."""
    vox = np.floor(np.asarray(points, np.float64) / voxel).astype(np.int64)
    _, inv, counts = np.unique(vox, axis=0, return_inverse=True,
                               return_counts=True)
    # centroid of each voxel
    sums = np.zeros((counts.size, 3), np.float64)
    np.add.at(sums, inv, points)
    centroids = sums / counts[:, None]
    # per-point squared distance to its own voxel centroid
    d2 = ((points - centroids[inv]) ** 2).sum(axis=1)
    # group points by voxel, take the closest-to-centroid one from each group
    order = np.lexsort((d2, inv))                 # sort by voxel, then distance
    first = np.concatenate([[0], np.cumsum(counts)[:-1]])
    return np.sort(order[first])

# ── Feature cache (compute once per file, reuse every epoch) ─────────────────
# Pipeline per file: load → voxel downsample (1 cm) → compute 7-ch features.
# Features are computed AFTER downsampling so surface normals (kNN=16) are
# estimated at the working resolution, and it is ~6× faster on iPhone scans.
FEATURE_CACHE = {}

def get_features(path):
    """Return (points[N,3], features[N,7], labels[N]) for a file —
    voxel-downsampled if CONFIG["voxel_size"] > 0 — cached after first call."""
    if path not in FEATURE_CACHE:
        pts, lbl = load_pointcloud(path)
        lbl = np.zeros(len(pts), dtype=np.int64) if lbl is None else lbl
        lbl = np.clip(lbl, 0, NUM_CLASSES - 1).astype(np.int64)

        v = CONFIG.get("voxel_size", 0)
        if v and v > 0 and len(pts) > 0:
            keep = voxel_downsample_idx(pts, v)
            log.info(f"  voxel {v*100:.0f}cm: {len(pts):,} → {len(keep):,} pts "
                     f"({os.path.basename(path)})")
            pts, lbl = pts[keep], lbl[keep]

        feat = make_features(pts)
        FEATURE_CACHE[path] = (pts, feat, lbl)
    return FEATURE_CACHE[path]

log.info("Pre-computing features for train + val files (one-time cost)...")
for f in tqdm(TRAIN_FILES + VAL_FILES, desc="features"):
    get_features(f)
log.info("Done.")


2026-07-11 03:02:40,126 | INFO | Pre-computing features for train + val files (one-time cost)...


features:   0%|          | 0/39 [00:00<?, ?it/s]

2026-07-11 03:02:40,458 | INFO |   voxel 1cm: 380,210 → 302,755 pts (sample_data_0039.las)
2026-07-11 03:02:41,034 | INFO |   voxel 1cm: 572,459 → 458,389 pts (sample_data_0027.las)
2026-07-11 03:02:41,477 | INFO |   voxel 1cm: 381,523 → 286,623 pts (sample_data_0025.las)
2026-07-11 03:02:41,948 | INFO |   voxel 1cm: 474,218 → 362,541 pts (sample_data_0021.las)
2026-07-11 03:02:42,466 | INFO |   voxel 1cm: 491,964 → 403,897 pts (sample_data_007.las)
2026-07-11 03:02:43,083 | INFO |   voxel 1cm: 576,065 → 476,564 pts (sample_data_0023.las)
2026-07-11 03:02:43,717 | INFO |   voxel 1cm: 559,689 → 453,807 pts (sample_data_0018.las)
2026-07-11 03:02:44,334 | INFO |   voxel 1cm: 561,705 → 437,827 pts (sample_data_0024.las)
2026-07-11 03:02:44,960 | INFO |   voxel 1cm: 564,069 → 453,269 pts (sample_data_0043.las)
2026-07-11 03:02:45,469 | INFO |   voxel 1cm: 446,224 → 339,343 pts (sample_data_0038.las)
2026-07-11 03:02:46,056 | INFO |   voxel 1cm: 580,903 → 438,845 pts (sample_data_0034.las)


In [5]:
# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_miou(true_labels, pred_labels, num_classes):
    """Compute mean Intersection-over-Union across all classes."""
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0


def compute_dice(true_labels, pred_labels, num_classes):
    """Compute mean Dice Score across all classes."""
    dices = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            # Dice Formula: 2*TP / (2*TP + FP + FN)
            dices.append((2 * tp) / (2 * tp + fp + fn))
    return float(np.mean(dices)) if dices else 0.0

In [6]:
# ── MLflow setup ─────────────────────────────────────────────────────────────
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")


2026-07-11 03:03:03,583 | INFO | MLflow tracking: http://localhost:5000


## Model Architecture

In [7]:
MODEL_NAME = "PointTransformerV2"

from torch_cluster import knn as tc_knn
from torch_scatter import scatter_softmax, scatter_add, scatter_max, scatter_mean

class GVA(nn.Module):
    "Grouped Vector Attention — PTv2 core attention layer."
    def __init__(self, ch, g=6, k=16):
        super().__init__()
        assert ch % g == 0
        self.k, self.g, self.gc = k, g, ch // g
        self.q  = nn.Linear(ch, ch); self.kk = nn.Linear(ch, ch); self.v = nn.Linear(ch, ch)
        self.pm = nn.Sequential(nn.Linear(3,ch), nn.ReLU(), nn.Linear(ch,ch))
        self.pb = nn.Sequential(nn.Linear(3,ch), nn.ReLU(), nn.Linear(ch,ch))
        self.w  = nn.Sequential(nn.Linear(ch,ch), nn.ReLU(), nn.Linear(ch,g))
    def forward(self, x, pos, batch):
        e    = tc_knn(pos, pos, self.k, batch, batch)
        c, nb = e[0], e[1]
        dp   = pos[c] - pos[nb]
        pb   = self.pb(dp)
        rel  = (self.q(x)[c] - self.kk(x)[nb]) * self.pm(dp) + pb
        wt   = scatter_softmax(self.w(rel), c, dim=0)
        vg   = (self.v(x)[nb] + pb).view(-1, self.g, self.gc)
        out  = scatter_add(vg * wt.unsqueeze(-1), c, dim=0, dim_size=x.size(0))
        return out.view(-1, self.g * self.gc)

class Block(nn.Module):
    def __init__(self, ch, g=6, k=16):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(ch), nn.LayerNorm(ch)
        self.attn = GVA(ch, g, k)
        self.mlp  = nn.Sequential(nn.Linear(ch,ch*2), nn.ReLU(), nn.Linear(ch*2,ch))
    def forward(self, x, pos, batch):
        x = x + self.attn(self.n1(x), pos, batch)
        return x + self.mlp(self.n2(x))

class GridPool(nn.Module):
    def __init__(self, i, o, grid):
        super().__init__()
        self.grid = grid
        self.proj = nn.Sequential(nn.Linear(i,o), nn.ReLU())
    def forward(self, x, pos, batch):
        vox = torch.floor(pos/self.grid).long()
        key = torch.cat([batch.unsqueeze(1), vox], 1)
        _, cl = torch.unique(key, dim=0, return_inverse=True)
        xp, _ = scatter_max(self.proj(x), cl, dim=0)
        return xp, scatter_mean(pos,cl,dim=0), scatter_max(batch,cl,dim=0)[0], cl

class PTv2Seg(nn.Module):
    def __init__(self, nc=NUM_CLASSES, k=16, dims=(48,96,192), g=6):
        super().__init__()
        d = dims
        self.embed = nn.Sequential(nn.Linear(7,d[0]),nn.ReLU(),nn.Linear(d[0],d[0]))
        self.e1=Block(d[0],g,k); self.p1=GridPool(d[0],d[1],0.08)
        self.e2=Block(d[1],g,k); self.p2=GridPool(d[1],d[2],0.16)
        self.e3=Block(d[2],g,k)
        self.u2=nn.Sequential(nn.Linear(d[2]+d[1],d[1]),nn.ReLU()); self.d2=Block(d[1],g,k)
        self.u1=nn.Sequential(nn.Linear(d[1]+d[0],d[0]),nn.ReLU()); self.d1=Block(d[0],g,k)
        self.head=nn.Sequential(nn.LayerNorm(d[0]),nn.Linear(d[0],128),nn.ReLU(),
                                nn.Dropout(0.4),nn.Linear(128,nc))
    def forward(self, x):
        B,C,N = x.shape
        flat  = x.permute(0,2,1).reshape(-1,C).contiguous()
        p0    = flat[:,:3].contiguous()
        b0    = torch.arange(B,device=x.device).repeat_interleave(N)
        h0=self.e1(self.embed(flat),p0,b0)
        h1,p1,b1,c1=self.p1(h0,p0,b0); h1=self.e2(h1,p1,b1)
        h2,p2,b2,c2=self.p2(h1,p1,b1); h2=self.e3(h2,p2,b2)
        u1=self.d2(self.u2(torch.cat([h1,h2[c2]],1)),p1,b1)
        u0=self.d1(self.u1(torch.cat([h0,u1[c1]],1)),p0,b0)
        return self.head(u0).view(B,N,-1).permute(0,2,1)

model = PTv2Seg()
log.info(f"PTv2 parameters: {sum(p.numel() for p in model.parameters()):,}")


2026-07-11 03:03:03,621 | INFO | PTv2 parameters: 679,424


## Dataset & DataLoader

In [8]:
# ── Dataset: one random chunk per call ───────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class PointCloudDataset(Dataset):
    """Returns one fixed-size sphere-crop chunk per item.
    This class only handles data — no training logic here."""
    def __init__(self, files, augment=True):
        self.files   = list(files)
        self.augment = augment
        self.N       = CONFIG["num_points"]
        self.chunks  = CONFIG["chunks_per_cloud"]

    def __len__(self):
        return len(self.files) * self.chunks

    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        pts, feat, lbl = get_features(path)
        n = len(feat)

        # RNG: random for training, deterministic per-item for validation.
        # Without this, the val set samples DIFFERENT chunks every epoch,
        # which makes val mIoU noisy and best-checkpoint selection unreliable.
        rng = np.random if self.augment else np.random.RandomState(idx)

        # pick N nearest points around a (random / fixed) seed point
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            seed = rng.randint(n)
            dist = ((feat[:, :3] - feat[seed, :3]) ** 2).sum(1)
            chosen = np.argpartition(dist, self.N - 1)[:self.N]

        x = feat[chosen].copy()    # (N, 7)
        y = lbl[chosen].copy()     # (N,)

        # data augmentation: random rotation around vertical axis
        if self.augment:
            angle = np.random.uniform(0, 2 * np.pi)
            c, s  = np.cos(angle), np.sin(angle)
            R     = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float32)
            x[:, :3]  = x[:, :3]  @ R.T   # rotate xyz
            x[:, 4:7] = x[:, 4:7] @ R.T   # rotate normals

        # shape: (channels, points) for Conv1d / attention layers
        return torch.from_numpy(x.T), torch.from_numpy(y)


## Training

In [9]:
# ── Training loop with periodic checkpointing ────────────────────────────────
# Every SAVE_EVERY epochs:  saves model + optimizer state → resume after power cut
# Always keeps:             the BEST val-mIoU model as a separate file
# On resume:                loads the LATEST periodic checkpoint and continues
# Disk policy:              only keeps the last 2 periodic checkpoints (saves space)

SAVE_EVERY = 10    # save a checkpoint every this many epochs

def train_model(model, model_name):
    """Train the model with periodic saving. Resumes automatically if a
    checkpoint exists. Returns the model loaded with the best weights."""

    ckpt_dir  = CONFIG["checkpoint_dir"]
    best_path = os.path.join(ckpt_dir, f"{model_name}_best.pth")

    # ── helper: find the latest periodic checkpoint ───────────────────────────
    def latest_periodic():
        """Return (epoch, path) of the most recent periodic checkpoint, or (0, None)."""
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))   # sorted alphabetically = epoch order
        if not files:
            return 0, None
        # extract epoch number from filename, pick the highest
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        return epoch_of(files[-1]), files[-1]

    # ── helper: delete old periodic checkpoints, keep only the last 2 ─────────
    def prune_old_checkpoints(keep=2):
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        for old in files[:-keep]:
            try:    os.remove(old); log.info(f"Removed old checkpoint: {old}")
            except: pass

    # ── data loaders ──────────────────────────────────────────────────────────
    train_loader = DataLoader(
        PointCloudDataset(TRAIN_FILES, augment=True),
        batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
    val_loader = DataLoader(
        PointCloudDataset(VAL_FILES, augment=False),
        batch_size=CONFIG["batch_size"], shuffle=False)

    # move model to GPU FIRST so optimizer tracks GPU parameters from the start
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max",                      # maximize val mIoU
        factor=CONFIG["sched_factor"],
        patience=CONFIG["sched_patience"],
        min_lr=CONFIG["min_lr"])
    loss_fn   = nn.CrossEntropyLoss()

    # ── try to resume from the latest periodic checkpoint ────────────────────
    start_epoch = 1
    best_miou   = -1.0
    no_improve  = 0
    history     = []

    resume_epoch, resume_path = latest_periodic()
    if resume_path:
        log.info(f"Resuming from {resume_path} (epoch {resume_epoch})")
        ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        # move optimizer state tensors to the same device as the model
        for state in optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(DEVICE)
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_miou   = ckpt.get("best_miou",   -1.0)
        no_improve  = ckpt.get("no_improve",   0)
        history     = ckpt.get("history",      [])
        log.info(f"Resumed: start_epoch={start_epoch}  best_miou={best_miou:.4f}")
    else:
        log.info(f"No checkpoint found — training from scratch.")

    # ── if already fully trained, just load best and return ──────────────────
    if start_epoch > CONFIG["epochs"]:
        log.info("Training already complete. Loading best model.")
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        return model

    run_name = f"{model_name}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:

        if MLFLOW_OK:
            mlflow.log_params({
                "model"       : model_name,
                "epochs"      : CONFIG["epochs"],
                "batch_size"  : CONFIG["batch_size"],
                "num_points"  : CONFIG["num_points"],
                "lr"          : CONFIG["lr"],
                "start_epoch" : start_epoch,
            })

        accumulation_steps = CONFIG.get("grad_accum", 4)  # gradient accumulation steps

        for epoch in range(start_epoch, CONFIG["epochs"] + 1):

            # ── train one epoch ───────────────────────────────────────────────
            model.train()
            optimizer.zero_grad()
            
            train_loss_sum = 0.0
            train_correct = 0
            train_total = 0
            train_true, train_pred = [], []

            for i, (x, y) in enumerate(train_loader):
                x, y = x.to(DEVICE), y.to(DEVICE)
                
                # Forward
                out = model(x)
                loss = loss_fn(out, y)
                
                # Tracking metrics (before scaling loss)
                train_loss_sum += loss.item() * x.size(0)
                preds = out.argmax(dim=1)
                train_correct += (preds == y).sum().item()
                train_total += y.numel()
                train_pred.extend(preds.detach().cpu().numpy().ravel())
                train_true.extend(y.detach().cpu().numpy().ravel())

                # Gradient accumulation
                loss = loss / accumulation_steps
                loss.backward()

                if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                    optimizer.step()
                    optimizer.zero_grad()

            epoch_train_loss = train_loss_sum / len(train_loader.dataset)
            epoch_train_acc = train_correct / train_total
            train_miou = compute_miou(np.array(train_true),
                                      np.array(train_pred), NUM_CLASSES)

            # ── validate ─────────────────────────────────────────────────────
            model.eval()
            val_loss_sum = 0.0
            val_correct = 0
            val_total = 0
            all_true, all_pred = [], []
            
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    
                    out = model(x)
                    loss = loss_fn(out, y)
                    
                    val_loss_sum += loss.item() * x.size(0)
                    preds = out.argmax(dim=1)
                    val_correct += (preds == y).sum().item()
                    val_total += y.numel()

                    all_pred.extend(preds.cpu().numpy().ravel())
                    all_true.extend(y.cpu().numpy().ravel())

            epoch_val_loss = val_loss_sum / len(val_loader.dataset)
            epoch_val_acc = val_correct / val_total
            
            # Compute mIoU and Dice (validation)
            val_miou = compute_miou(np.array(all_true), np.array(all_pred), NUM_CLASSES)
            val_dice = compute_dice(np.array(all_true), np.array(all_pred), NUM_CLASSES)
            
            # step the LR scheduler on validation mIoU
            scheduler.step(val_miou)
            current_lr = optimizer.param_groups[0]["lr"]

            history.append({"epoch": epoch,
                            "train_miou": train_miou, "val_miou": val_miou,
                            "train_loss": epoch_train_loss, "val_loss": epoch_val_loss,
                            "lr": current_lr})
            
            # ── Logging to Console ───────────────────────────────────────────
            log.info(
                f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
                f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | "
                f"Train Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f} | "
                f"Train mIoU: {train_miou:.4f} | Val mIoU: {val_miou:.4f} | "
                f"Val Dice: {val_dice:.4f} | LR: {current_lr:.2e}"
            )

            # ── Logging to MLflow ────────────────────────────────────────────
            if MLFLOW_OK:
                mlflow.log_metrics({
                    "train_loss": epoch_train_loss,
                    "val_loss": epoch_val_loss,
                    "train_acc": epoch_train_acc,
                    "val_acc": epoch_val_acc,
                    "train_miou": train_miou,
                    "val_miou": val_miou,
                    "val_dice": val_dice,
                    "lr": current_lr
                }, step=epoch)

            # ── update best model ─────────────────────────────────────────────
            if val_miou > best_miou + 1e-4:
                best_miou  = val_miou
                no_improve = 0
                torch.save({"model_state": model.state_dict(),
                            "best_val_miou": best_miou}, best_path)
                log.info(f"  ✓ New best mIoU {best_miou:.4f} → saved {best_path}")
            else:
                no_improve += 1

            # ── periodic checkpoint (every SAVE_EVERY epochs) ─────────────────
            if epoch % SAVE_EVERY == 0 or epoch == CONFIG["epochs"]:
                periodic_path = os.path.join(ckpt_dir,
                                             f"{model_name}_epoch_{epoch:04d}.pth")
                torch.save({
                    "model_state"     : model.state_dict(),
                    "optimizer_state" : optimizer.state_dict(),
                    "scheduler_state" : scheduler.state_dict(),
                    "epoch"           : epoch,
                    "best_miou"       : best_miou,
                    "no_improve"      : no_improve,
                    "history"         : history,
                }, periodic_path)
                log.info(f"  💾 Periodic checkpoint saved → {periodic_path}")
                prune_old_checkpoints(keep=2)   # keep only last 2 periodic files

            # ── early stopping ────────────────────────────────────────────────
            if no_improve >= CONFIG["patience"]:
                log.info(f"Early stop at epoch {epoch} "
                         f"(no improvement for {CONFIG['patience']} epochs)")
                break

        if MLFLOW_OK:
            mlflow.log_metric("best_val_miou", best_miou)

    # ── load best weights before returning ────────────────────────────────────
    if os.path.exists(best_path):
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        log.info(f"Loaded best model (mIoU={best_miou:.4f}) from {best_path}")
    model.to("cpu")
    return model


In [ ]:
model = train_model(model, MODEL_NAME)

2026-07-11 03:03:04,300 | INFO | No checkpoint found — training from scratch.
2026-07-11 03:04:27,069 | INFO | Epoch   1/150 | Train Loss: 0.6707 | Val Loss: 0.7004 | Train Acc: 0.6050 | Val Acc: 0.5910 | Train mIoU: 0.3539 | Val mIoU: 0.2955 | Val Dice: 0.3715 | LR: 2.00e-04
2026-07-11 03:04:27,112 | INFO |   ✓ New best mIoU 0.2955 → saved ../checkpoints_ptv2_voxel/PointTransformerV2_best.pth
2026-07-11 03:05:54,901 | INFO | Epoch   2/150 | Train Loss: 0.6371 | Val Loss: 0.6849 | Train Acc: 0.6658 | Val Acc: 0.5910 | Train mIoU: 0.3448 | Val mIoU: 0.2955 | Val Dice: 0.3715 | LR: 2.00e-04
2026-07-11 03:07:23,456 | INFO | Epoch   3/150 | Train Loss: 0.6148 | Val Loss: 0.7085 | Train Acc: 0.6837 | Val Acc: 0.5995 | Train mIoU: 0.4029 | Val mIoU: 0.3084 | Val Dice: 0.3938 | LR: 2.00e-04
2026-07-11 03:07:23,568 | INFO |   ✓ New best mIoU 0.3084 → saved ../checkpoints_ptv2_voxel/PointTransformerV2_best.pth
2026-07-11 03:08:52,482 | INFO | Epoch   4/150 | Train Loss: 0.5722 | Val Loss: 0.620

In [ ]:
# ── Training curves: Train vs Val mIoU and Loss ──────────────────────────────
# Loads `history` from the latest periodic checkpoint and plots the curves.
import glob, re
import matplotlib.pyplot as plt

def load_history():
    pattern = os.path.join(CONFIG["checkpoint_dir"], f"{MODEL_NAME}_epoch_*.pth")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError("No periodic checkpoint found — train first.")
    latest = max(files, key=lambda p: int(re.search(r"epoch_(\d+)", p).group(1)))
    ckpt = torch.load(latest, map_location="cpu", weights_only=False)
    return ckpt.get("history", []), latest

history, ckpt_file = load_history()
print(f"Loaded history ({len(history)} epochs) from {ckpt_file}")

epochs     = [h["epoch"] for h in history]
train_miou = [h.get("train_miou") for h in history]
val_miou   = [h["val_miou"] for h in history]
train_loss = [h.get("train_loss") for h in history]
val_loss   = [h["val_loss"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# mIoU curve (older checkpoints may not have train_miou — skip Nones)
if any(v is not None for v in train_miou):
    axes[0].plot(epochs, train_miou, label="Train mIoU")
axes[0].plot(epochs, val_miou, label="Val mIoU")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("mIoU")
axes[0].set_title("Train vs Val mIoU"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss curve
if any(v is not None for v in train_loss):
    axes[1].plot(epochs, train_loss, label="Train Loss")
axes[1].plot(epochs, val_loss, label="Val Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].set_title("Train vs Val Loss"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Visualization helper

In [ ]:
# ── Visualization: Open3D ────────────────────────────────────────────────────
# Green = target (wood powder)   |   Red = others (environment)
def visualize_segmentation(points, predictions, title="Segmentation"):
    """Open an Open3D window showing the segmentation result."""
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]   # green
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]   # red

    pcd = o3d.geometry.PointCloud()
    # centre the cloud so it appears at the origin
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # XYZ axes
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)

    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)


## Inference helper

In [ ]:
# ── Full-cloud inference ──────────────────────────────────────────────────────
from scipy.spatial import cKDTree as _cKDTree

@torch.no_grad()
def predict_full_cloud(model, path):
    """Label EVERY point of the ORIGINAL full-resolution cloud.
    Steps: predict on the voxel-downsampled cloud (fast, more context per chunk),
    then propagate labels to the full cloud via 1-nearest-neighbour — same
    label-propagation idea as the DBSCAN notebook. Returns full-res
    (points, predictions, ground-truth labels)."""
    model.eval()
    pts, feat, lbl = get_features(path)      # voxel-downsampled (or full if off)
    n    = len(feat)
    N    = CONFIG["num_points"]

    # random order so every point is covered
    perm = np.random.RandomState(0).permutation(n)
    pad  = (N - n % N) % N
    if pad:
        perm = np.concatenate([perm, perm[:pad]])
    chunks = perm.reshape(-1, N)

    preds = np.zeros(n, dtype=np.int64)
    model.to(DEVICE)
    for start in range(0, len(chunks), CONFIG["batch_size"]):
        cid = chunks[start : start + CONFIG["batch_size"]]
        x   = torch.from_numpy(feat[cid].transpose(0, 2, 1)).to(DEVICE)
        out = model(x).argmax(dim=1).cpu().numpy()
        preds[cid.ravel()] = out.ravel()
    model.to("cpu")

    # ── propagate to the original full-resolution cloud ──────────────────────
    if CONFIG.get("voxel_size", 0) > 0:
        pts_full, lbl_full = load_pointcloud(path)
        lbl_full = (np.zeros(len(pts_full), dtype=np.int64)
                    if lbl_full is None
                    else np.clip(lbl_full, 0, NUM_CLASSES - 1).astype(np.int64))
        _, nn_idx = _cKDTree(pts).query(pts_full, k=1)
        return pts_full, preds[nn_idx], lbl_full

    return pts, preds, lbl


## Volume Estimation (TIN)

Segmentation-এর পরে target points থেকে volume বের করা হয়।

**Pipeline (সব `vol_TIN()`-এর ভেতরে):**
1. **SOR** — noisy বিচ্ছিন্ন point বাদ (statistical distance filter)
2. **DBSCAN largest cluster** — stray mislabeled points বাদ
3. **TIN integration** — 2D Delaunay → Σ Area₂D(triangle) × mean height
   - Floor baseline = scanned z-এর `floor_pct` percentile (outer-shell scan-এ আলাদা floor point থাকে না)
   - কোনো alpha clipping নেই — DBSCAN-ই stray point সামলায়; clipping করলে কিনারার বৈধ triangle কেটে গিয়ে volume কমে যেত
4. **Bootstrap CI** — ৮০ বার subsample করে std + 95% confidence interval


In [ ]:
# ── Volume Estimation: TIN-based volumetric integration ─────────────────────
from scipy.spatial import Delaunay, cKDTree


def sor_filter(pts, k=16, std_ratio=2.0):
    """Statistical Outlier Removal: mean-kNN-distance > mean + std_ratio*std
    হলে সেই point বাদ। Returns (filtered_pts, kept_mask)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) <= k + 1:
        return pts, np.ones(len(pts), bool)
    tree = cKDTree(pts)
    d, _ = tree.query(pts, k=k + 1)          # col-0 = নিজেই (দূরত্ব 0)
    mean_dist = d[:, 1:].mean(axis=1)
    thr  = mean_dist.mean() + std_ratio * mean_dist.std()
    mask = mean_dist <= thr
    return pts[mask], mask


def extract_main_cluster(pts, eps=0.15, min_samples=8):
    """DBSCAN দিয়ে সবচেয়ে বড় connected cluster রাখা — stray points বাদ।
    এর পরে আর alpha clipping লাগে না।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < min_samples * 2:
        return pts
    try:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        labels = np.asarray(pcd.cluster_dbscan(eps=eps, min_points=min_samples))
        valid = labels[labels >= 0]
        if valid.size == 0:
            return pts
        main_lbl = np.bincount(valid).argmax()
        result = pts[labels == main_lbl]
        return result if len(result) >= min_samples else pts
    except Exception as e:
        log.warning(f"[TIN] cluster extraction failed ({e}) -> using all points")
        return pts


def tin_integrate(pts, floor_pct=0.5):
    """Delaunay TIN integration.
    floor baseline = scanned z-এর floor_pct percentile (pile-এর base প্রান্ত)।
    Volume = Σ Area2D(triangle) × mean_height(triangle)."""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 4:
        return float("nan")

    floor_z = np.percentile(pts[:, 2], floor_pct)
    h  = pts[:, 2] - floor_z
    xy = pts[:, :2]
    try:
        tri = Delaunay(xy)
    except Exception as e:
        log.warning(f"[TIN] Delaunay failed ({e})")
        return float("nan")

    t = tri.simplices
    p0, p1, p2 = xy[t[:, 0]], xy[t[:, 1]], xy[t[:, 2]]
    h0, h1, h2 = h[t[:, 0]], h[t[:, 1]], h[t[:, 2]]

    cross  = ((p1[:, 0] - p0[:, 0]) * (p2[:, 1] - p0[:, 1])
             - (p1[:, 1] - p0[:, 1]) * (p2[:, 0] - p0[:, 0]))
    area2d = np.abs(cross) / 2.0
    mean_h = (h0 + h1 + h2) / 3.0
    valid  = mean_h > 0
    return float(np.sum(area2d[valid] * mean_h[valid]))


def bootstrap_tin_confidence(pts, n_bootstrap=80, subsample_frac=0.85,
                             floor_pct=0.5, seed=0):
    """Bootstrap: বারবার subsample করে TIN চালিয়ে uncertainty মাপা।
    Returns {'std', 'ci_low', 'ci_high'} বা None।"""
    rng = np.random.RandomState(seed)
    n = len(pts)
    n_sub = max(10, int(n * subsample_frac))
    estimates = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n_sub, replace=False)
        v = tin_integrate(pts[idx], floor_pct=floor_pct)
        if np.isfinite(v) and v > 0:
            estimates.append(v)
    if len(estimates) < 5:
        return None
    arr = np.array(estimates)
    return {"std":     round(float(arr.std()), 5),
            "ci_low":  round(float(np.percentile(arr, 2.5)), 5),
            "ci_high": round(float(np.percentile(arr, 97.5)), 5)}


def vol_TIN(pts, sor_k=16, sor_std=2.0, cluster_eps=0.15,
            floor_pct=0.5, run_bootstrap=True, n_bootstrap=80):
    """সম্পূর্ণ volume pipeline: SOR → largest cluster → TIN [→ bootstrap CI].
    Returns (volume, confidence_dict_or_None). একক = pts-এর এককের ঘন (m³)।"""
    pts = np.asarray(pts, np.float64)
    if len(pts) < 10:
        return float("nan"), None

    pts, _ = sor_filter(pts, k=sor_k, std_ratio=sor_std)          # 1. SOR
    if len(pts) < 10:
        return float("nan"), None

    pts = extract_main_cluster(pts, eps=cluster_eps)               # 2. cluster
    if len(pts) < 10:
        return float("nan"), None

    volume = tin_integrate(pts, floor_pct=floor_pct)               # 3. TIN

    conf = None                                                    # 4. CI
    if run_bootstrap and len(pts) >= 20:
        conf = bootstrap_tin_confidence(pts, n_bootstrap=n_bootstrap,
                                        floor_pct=floor_pct)
    return float(volume), conf


## Testing  (held-out test files with labels)

In [ ]:
# ── Testing on held-out test files ───────────────────────────────────────────
log.info("=== TESTING ===")
test_mious = []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(model, path)
    miou = compute_miou(lbl, preds, NUM_CLASSES)
    test_mious.append(miou)
    log.info(f"  {name}: mIoU = {miou:.4f}")

log.info(f"Mean test mIoU: {np.mean(test_mious):.4f}")

# ── Visualize + volume for the first few test files ──────────────────────────
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud(model, path)

    # segmentation → vol_TIN → volume
    target_pts   = pts[preds == CONFIG["target_class"]]
    volume, conf = vol_TIN(target_pts)

    n_tot = len(pts)
    n_tgt = len(target_pts)
    print(f"\n{'─'*55}")
    print(f"  File          : {name}")
    print(f"  Total Points  : {n_tot:,}")
    print(f"  Target Points : {n_tgt:,}")
    print(f"  Other Points  : {n_tot - n_tgt:,}")
    print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
    print(f"  Est. Volume   : {volume:.4f} m³")
    if conf:
        print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
              f"  (±{conf['std']:.4f})")
    print(f"{'─'*55}")

    visualize_segmentation(pts, preds,
                           title=f"{MODEL_NAME} | {name} | V={volume:.4f} m³")


2026-07-09 17:52:48,091 | INFO | === TESTING ===
2026-07-09 17:52:54,593 | INFO |   sample_data_0013: mIoU = 0.8439
2026-07-09 17:53:02,227 | INFO |   sample_data_0020: mIoU = 0.8777
2026-07-09 17:53:12,269 | INFO |   sample_data_0017: mIoU = 0.8980
2026-07-09 17:53:19,995 | INFO |   sample_data_0012: mIoU = 0.8660
2026-07-09 17:53:27,749 | INFO |   sample_data_0015: mIoU = 0.9026
2026-07-09 17:53:32,089 | INFO |   sample_data_0031: mIoU = 0.7954
2026-07-09 17:53:32,090 | INFO | Mean test mIoU: 0.8639


## Final Inference  (`data/test`, no labels)

In [ ]:
# ── Final inference on data/test (no labels needed) ──────────────────────────
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, _ = predict_full_cloud(model, path)

        # segmentation → vol_TIN → volume
        target_pts   = pts[preds == CONFIG["target_class"]]
        volume, conf = vol_TIN(target_pts)

        n_tot = len(pts)
        n_tgt = len(target_pts)
        print(f"\n{'─'*55}")
        print(f"  File          : {name}")
        print(f"  Total Points  : {n_tot:,}")
        print(f"  Target Points : {n_tgt:,}")
        print(f"  Other Points  : {n_tot - n_tgt:,}")
        print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
        print(f"  Est. Volume   : {volume:.4f} m³")
        if conf:
            print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
                  f"  (±{conf['std']:.4f})")
        print(f"{'─'*55}")

        visualize_segmentation(pts, preds,
                               title=f"INFERENCE | {MODEL_NAME} | {name} | "
                                     f"V={volume:.4f} m³")
else:
    log.info("data/test is empty — skipping inference.")


2026-07-09 19:37:59,661 | INFO | === FINAL INFERENCE ===
2026-07-09 19:38:07,052 | INFO |   sample_data_0046: target=460,740 / total=601,991
2026-07-09 19:38:34,180 | INFO |   sample_data_0047: target=530,481 / total=678,899
2026-07-09 19:39:08,196 | INFO |   sample_data_0048: target=330,399 / total=506,292
2026-07-09 19:44:54,788 | INFO |   sample_data_0049: target=491,013 / total=717,491
2026-07-09 19:45:33,034 | INFO |   sample_data_0050: target=312,172 / total=453,747
2026-07-09 19:48:11,146 | INFO |   sample_data_0051: target=440,988 / total=727,068
2026-07-09 19:49:41,669 | INFO |   sample_data_0052: target=304,466 / total=501,649
2026-07-09 19:50:40,645 | INFO |   sample_data_0053: target=399,249 / total=606,372
2026-07-09 19:51:46,459 | INFO |   sample_data_0054: target=353,263 / total=552,492
2026-07-09 19:52:53,407 | INFO |   sample_data_0055: target=361,371 / total=459,550
2026-07-09 19:54:14,116 | INFO |   sample_data_0056: target=482,276 / total=773,541
2026-07-09 19:55:25

In [ ]:
import torch
torch.version.cuda

'11.8'